# PhonePe Transaction Insights
## Notebook 2: Data Extraction and Transformation

### Objective
This notebook extracts data from the PhonePe Pulse JSON dataset and transforms it into structured tabular form for SQL loading.

### Output
Processed CSV files for:
- Aggregated Transaction
- Aggregated User
- Map Transaction
- Map User
- Top Transaction
- Top User
- Aggregated Insurance (if available)

In [1]:
import os
import json
import pandas as pd
import warnings

In [2]:
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

In [3]:
PROJECT_ROOT = os.path.abspath(".")
RAW_DATA_PATH = os.path.join(PROJECT_ROOT, "data", "pulse-master", "data")
PROCESSED_DATA_PATH = os.path.join(PROJECT_ROOT, "data", "processed")

os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

print("Raw Data Path:", RAW_DATA_PATH)
print("Processed Data Path:", PROCESSED_DATA_PATH)

Raw Data Path: C:\Users\cheta\PhonePe_Project\data\pulse-master\data
Processed Data Path: C:\Users\cheta\PhonePe_Project\data\processed


In [4]:
def clean_state_name(state_name):
    return state_name.replace("-", " ").title()

def load_json(file_path):
    with open(file_path, "r") as f:
        return json.load(f)

## Extract Aggregated Transaction Data

In [5]:
def extract_aggregated_transaction(base_path):
    records = []
    path = os.path.join(base_path, "aggregated", "transaction", "country", "india", "state")

    for state in os.listdir(path):
        state_path = os.path.join(path, state)
        if not os.path.isdir(state_path):
            continue

        for year in os.listdir(state_path):
            year_path = os.path.join(state_path, year)

            for file in os.listdir(year_path):
                if file.endswith(".json"):
                    quarter = int(file.replace(".json", ""))
                    file_path = os.path.join(year_path, file)
                    data = load_json(file_path)

                    transaction_data = data.get("data", {}).get("transactionData", [])

                    for item in transaction_data:
                        records.append({
                            "state": clean_state_name(state),
                            "year": int(year),
                            "quarter": quarter,
                            "transaction_type": item.get("name"),
                            "transaction_count": item.get("paymentInstruments", [{}])[0].get("count", 0),
                            "transaction_amount": item.get("paymentInstruments", [{}])[0].get("amount", 0)
                        })

    return pd.DataFrame(records)

df_aggregated_transaction = extract_aggregated_transaction(RAW_DATA_PATH)
df_aggregated_transaction.head()

,state,year,quarter,transaction_type,transaction_count,transaction_amount
0,Andaman & Nicobar Islands,2018,1,Recharge & bill payments,4200,1.845307e+06
1,Andaman & Nicobar Islands,2018,1,Peer-to-peer payments,1871,1.213866e+07
2,Andaman & Nicobar Islands,2018,1,Merchant payments,298,4.525072e+05
3,Andaman & Nicobar Islands,2018,1,Financial Services,33,1.060142e+04
4,Andaman & Nicobar Islands,2018,1,Others,256,1.846899e+05


In [6]:
df_aggregated_transaction.shape

(5034, 6)

## Extract Aggregated User Data

In [7]:
def extract_aggregated_user(base_path):
    records = []
    path = os.path.join(base_path, "aggregated", "user", "country", "india", "state")

    for state in os.listdir(path):
        state_path = os.path.join(path, state)
        if not os.path.isdir(state_path):
            continue

        for year in os.listdir(state_path):
            year_path = os.path.join(state_path, year)

            for file in os.listdir(year_path):
                if file.endswith(".json"):
                    quarter = int(file.replace(".json", ""))
                    file_path = os.path.join(year_path, file)
                    data = load_json(file_path)

                    users_by_device = data.get("data", {}).get("usersByDevice", [])

                    if users_by_device:
                        for item in users_by_device:
                            records.append({
                                "state": clean_state_name(state),
                                "year": int(year),
                                "quarter": quarter,
                                "brand": item.get("brand"),
                                "count": item.get("count", 0),
                                "percentage": item.get("percentage", 0)
                            })
                    else:
                        records.append({
                            "state": clean_state_name(state),
                            "year": int(year),
                            "quarter": quarter,
                            "brand": None,
                            "count": 0,
                            "percentage": 0
                        })

    return pd.DataFrame(records)

df_aggregated_user = extract_aggregated_user(RAW_DATA_PATH)
df_aggregated_user.head()

,state,year,quarter,brand,count,percentage
0,Andaman & Nicobar Islands,2018,1,Xiaomi,1665,0.247033
1,Andaman & Nicobar Islands,2018,1,Samsung,1445,0.214392
2,Andaman & Nicobar Islands,2018,1,Vivo,982,0.145697
3,Andaman & Nicobar Islands,2018,1,Oppo,501,0.074332
4,Andaman & Nicobar Islands,2018,1,OnePlus,332,0.049258


In [8]:
df_aggregated_user.shape

(7128, 6)

## Extract Map Transaction Data

In [9]:
def extract_map_transaction(base_path):
    records = []
    path = os.path.join(base_path, "map", "transaction", "hover", "country", "india", "state")

    for state in os.listdir(path):
        state_path = os.path.join(path, state)
        if not os.path.isdir(state_path):
            continue

        for year in os.listdir(state_path):
            year_path = os.path.join(state_path, year)

            for file in os.listdir(year_path):
                if file.endswith(".json"):
                    quarter = int(file.replace(".json", ""))
                    file_path = os.path.join(year_path, file)
                    data = load_json(file_path)

                    hover_data_list = data.get("data", {}).get("hoverDataList", [])

                    for item in hover_data_list:
                        records.append({
                            "state": clean_state_name(state),
                            "year": int(year),
                            "quarter": quarter,
                            "district": item.get("name"),
                            "transaction_count": item.get("metric", [{}])[0].get("count", 0),
                            "transaction_amount": item.get("metric", [{}])[0].get("amount", 0)
                        })

    return pd.DataFrame(records)

df_map_transaction = extract_map_transaction(RAW_DATA_PATH)
df_map_transaction.head()

,state,year,quarter,district,transaction_count,transaction_amount
0,Andaman & Nicobar Islands,2018,1,north and middle andaman district,442,9.316631e+05
1,Andaman & Nicobar Islands,2018,1,south andaman district,5688,1.256025e+07
2,Andaman & Nicobar Islands,2018,1,nicobars district,528,1.139849e+06
3,Andaman & Nicobar Islands,2018,2,north and middle andaman district,825,1.317863e+06
4,Andaman & Nicobar Islands,2018,2,south andaman district,9395,2.394824e+07


In [10]:
df_map_transaction.shape

(20604, 6)

## Extract Map User Data

In [11]:
def extract_map_user(base_path):
    records = []
    path = os.path.join(base_path, "map", "user", "hover", "country", "india", "state")

    for state in os.listdir(path):
        state_path = os.path.join(path, state)
        if not os.path.isdir(state_path):
            continue

        for year in os.listdir(state_path):
            year_path = os.path.join(state_path, year)

            for file in os.listdir(year_path):
                if file.endswith(".json"):
                    quarter = int(file.replace(".json", ""))
                    file_path = os.path.join(year_path, file)
                    data = load_json(file_path)

                    hover_data = data.get("data", {}).get("hoverData", {})

                    for district, values in hover_data.items():
                        records.append({
                            "state": clean_state_name(state),
                            "year": int(year),
                            "quarter": quarter,
                            "district": district,
                            "registered_users": values.get("registeredUsers", 0),
                            "app_opens": values.get("appOpens", 0)
                        })

    return pd.DataFrame(records)

df_map_user = extract_map_user(RAW_DATA_PATH)
df_map_user.head()

,state,year,quarter,district,registered_users,app_opens
0,Andaman & Nicobar Islands,2018,1,north and middle andaman district,632,0
1,Andaman & Nicobar Islands,2018,1,south andaman district,5846,0
2,Andaman & Nicobar Islands,2018,1,nicobars district,262,0
3,Andaman & Nicobar Islands,2018,2,north and middle andaman district,911,0
4,Andaman & Nicobar Islands,2018,2,south andaman district,8143,0


In [12]:
df_map_user.shape

(20608, 6)

## Extract Top Transaction Data

In [13]:
def extract_top_transaction(base_path):
    records = []
    path = os.path.join(base_path, "top", "transaction", "country", "india", "state")

    for state in os.listdir(path):
        state_path = os.path.join(path, state)
        if not os.path.isdir(state_path):
            continue

        for year in os.listdir(state_path):
            year_path = os.path.join(state_path, year)

            for file in os.listdir(year_path):
                if file.endswith(".json"):
                    quarter = int(file.replace(".json", ""))
                    file_path = os.path.join(year_path, file)
                    data = load_json(file_path)

                    districts = data.get("data", {}).get("districts", [])
                    pincodes = data.get("data", {}).get("pincodes", [])

                    for item in districts:
                        records.append({
                            "state": clean_state_name(state),
                            "year": int(year),
                            "quarter": quarter,
                            "entity_type": "district",
                            "entity_name": item.get("entityName"),
                            "transaction_count": item.get("metric", {}).get("count", 0),
                            "transaction_amount": item.get("metric", {}).get("amount", 0)
                        })

                    for item in pincodes:
                        records.append({
                            "state": clean_state_name(state),
                            "year": int(year),
                            "quarter": quarter,
                            "entity_type": "pincode",
                            "entity_name": item.get("entityName"),
                            "transaction_count": item.get("metric", {}).get("count", 0),
                            "transaction_amount": item.get("metric", {}).get("amount", 0)
                        })

    return pd.DataFrame(records)

df_top_transaction = extract_top_transaction(RAW_DATA_PATH)
df_top_transaction.head()

,state,year,quarter,entity_type,entity_name,transaction_count,transaction_amount
0,Andaman & Nicobar Islands,2018,1,district,south andaman,5688,1.256025e+07
1,Andaman & Nicobar Islands,2018,1,district,nicobars,528,1.139849e+06
2,Andaman & Nicobar Islands,2018,1,district,north and middle andaman,442,9.316631e+05
3,Andaman & Nicobar Islands,2018,1,pincode,744101,1622,2.769298e+06
4,Andaman & Nicobar Islands,2018,1,pincode,744103,1223,2.238042e+06


In [14]:
df_top_transaction.shape

(18295, 7)

## Extract Top User Data

In [15]:
def extract_top_user(base_path):
    records = []
    path = os.path.join(base_path, "top", "user", "country", "india", "state")

    for state in os.listdir(path):
        state_path = os.path.join(path, state)
        if not os.path.isdir(state_path):
            continue

        for year in os.listdir(state_path):
            year_path = os.path.join(state_path, year)

            for file in os.listdir(year_path):
                if file.endswith(".json"):
                    quarter = int(file.replace(".json", ""))
                    file_path = os.path.join(year_path, file)
                    data = load_json(file_path)

                    districts = data.get("data", {}).get("districts", [])
                    pincodes = data.get("data", {}).get("pincodes", [])

                    for item in districts:
                        records.append({
                            "state": clean_state_name(state),
                            "year": int(year),
                            "quarter": quarter,
                            "entity_type": "district",
                            "entity_name": item.get("name"),
                            "registered_users": item.get("registeredUsers", 0)
                        })

                    for item in pincodes:
                        records.append({
                            "state": clean_state_name(state),
                            "year": int(year),
                            "quarter": quarter,
                            "entity_type": "pincode",
                            "entity_name": item.get("name"),
                            "registered_users": item.get("registeredUsers", 0)
                        })

    return pd.DataFrame(records)

df_top_user = extract_top_user(RAW_DATA_PATH)
df_top_user.head()

,state,year,quarter,entity_type,entity_name,registered_users
0,Andaman & Nicobar Islands,2018,1,district,south andaman,5846
1,Andaman & Nicobar Islands,2018,1,district,north and middle andaman,632
2,Andaman & Nicobar Islands,2018,1,district,nicobars,262
3,Andaman & Nicobar Islands,2018,1,pincode,744103,1608
4,Andaman & Nicobar Islands,2018,1,pincode,744101,1108


In [16]:
df_top_user.shape

(18296, 6)

## Extract Aggregated Insurance Data

In [17]:
def extract_aggregated_insurance(base_path):
    records = []
    path = os.path.join(base_path, "aggregated", "insurance", "country", "india", "state")

    if not os.path.exists(path):
        return pd.DataFrame()

    for state in os.listdir(path):
        state_path = os.path.join(path, state)
        if not os.path.isdir(state_path):
            continue

        for year in os.listdir(state_path):
            year_path = os.path.join(state_path, year)

            for file in os.listdir(year_path):
                if file.endswith(".json"):
                    quarter = int(file.replace(".json", ""))
                    file_path = os.path.join(year_path, file)
                    data = load_json(file_path)

                    insurance_data = data.get("data", {}).get("transactionData", [])

                    for item in insurance_data:
                        records.append({
                            "state": clean_state_name(state),
                            "year": int(year),
                            "quarter": quarter,
                            "insurance_type": item.get("name"),
                            "transaction_count": item.get("paymentInstruments", [{}])[0].get("count", 0),
                            "transaction_amount": item.get("paymentInstruments", [{}])[0].get("amount", 0)
                        })

    return pd.DataFrame(records)

df_aggregated_insurance = extract_aggregated_insurance(RAW_DATA_PATH)
df_aggregated_insurance.head()

,state,year,quarter,insurance_type,transaction_count,transaction_amount
0,Andaman & Nicobar Islands,2020,2,Insurance,6,1360.0
1,Andaman & Nicobar Islands,2020,3,Insurance,41,15380.0
2,Andaman & Nicobar Islands,2020,4,Insurance,124,157975.0
3,Andaman & Nicobar Islands,2021,1,Insurance,225,244266.0
4,Andaman & Nicobar Islands,2021,2,Insurance,137,181504.0


In [18]:
df_aggregated_insurance.shape

(682, 6)

## Basic Data Validation

In [19]:
datasets = {
    "aggregated_transaction": df_aggregated_transaction,
    "aggregated_user": df_aggregated_user,
    "map_transaction": df_map_transaction,
    "map_user": df_map_user,
    "top_transaction": df_top_transaction,
    "top_user": df_top_user,
    "aggregated_insurance": df_aggregated_insurance
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Null Values:\n", df.isnull().sum())


aggregated_transaction
Shape: (5034, 6)
Null Values:
 state                 0
year                  0
quarter               0
transaction_type      0
transaction_count     0
transaction_amount    0
dtype: int64

aggregated_user
Shape: (7128, 6)
Null Values:
 state           0
year            0
quarter         0
brand         396
count           0
percentage      0
dtype: int64

map_transaction
Shape: (20604, 6)
Null Values:
 state                 0
year                  0
quarter               0
district              0
transaction_count     0
transaction_amount    0
dtype: int64

map_user
Shape: (20608, 6)
Null Values:
 state               0
year                0
quarter             0
district            0
registered_users    0
app_opens           0
dtype: int64

top_transaction
Shape: (18295, 7)
Null Values:
 state                 0
year                  0
quarter               0
entity_type           0
entity_name           2
transaction_count     0
transaction_amount    0
dtype: in

## Save Processed CSV Files

In [20]:
df_aggregated_transaction.to_csv(os.path.join(PROCESSED_DATA_PATH, "aggregated_transaction.csv"), index=False)
df_aggregated_user.to_csv(os.path.join(PROCESSED_DATA_PATH, "aggregated_user.csv"), index=False)
df_map_transaction.to_csv(os.path.join(PROCESSED_DATA_PATH, "map_transaction.csv"), index=False)
df_map_user.to_csv(os.path.join(PROCESSED_DATA_PATH, "map_user.csv"), index=False)
df_top_transaction.to_csv(os.path.join(PROCESSED_DATA_PATH, "top_transaction.csv"), index=False)
df_top_user.to_csv(os.path.join(PROCESSED_DATA_PATH, "top_user.csv"), index=False)

if not df_aggregated_insurance.empty:
    df_aggregated_insurance.to_csv(os.path.join(PROCESSED_DATA_PATH, "aggregated_insurance.csv"), index=False)

print("All processed CSV files saved successfully.")

All processed CSV files saved successfully.


## Notebook 2 Completion

At this stage:
- Raw JSON data has been extracted
- Structured DataFrames are created
- CSV files are saved in the processed folder

Next notebook:
Notebook 3 will connect to MySQL and create the required project tables.